> ⚠️ **WARNING — UNDER CONSTRUCTION**
>
> This notebook is a work in progress. **Use at your own risk.**
> It may contain errors, incomplete sections, or incorrect results.
> No guarantees are made about its accuracy or completeness.

# Galaxy Rotation Curves and Dark Matter
## Evidence for dark matter halos from HI 21 cm observations

**Data:** THINGS (The HI Nearby Galaxy Survey) — Walter et al. (2008), AJ 136, 2563  
**Galaxy:** NGC 2403 (Sc spiral, $d = 3.2$ Mpc)  
**References:**  
- Walter et al. (2008), AJ 136, 2563–2647. [arXiv:0810.2125](https://arxiv.org/abs/0810.2125) — THINGS survey  
- de Blok et al. (2008), AJ 136, 2648. [arXiv:0810.2100](https://arxiv.org/abs/0810.2100) — THINGS mass models  
- Navarro, Frenk & White (1997), ApJ 490, 493. [arXiv:astro-ph/9611107](https://arxiv.org/abs/astro-ph/9611107) — NFW halo profile  
- Rubin & Ford (1970), ApJ 159, 379 — seminal rotation curve paper (M31 and others)  
- Freeman (1970), ApJ 160, 811 — exponential disk model  
- Begeman (1989), A&A 223, 47 — HI observations and tilted-ring models  

**THINGS data portal:** [http://www.mpia.de/THINGS/Data.html](http://www.mpia.de/THINGS/Data.html)

**Key tools:** `numpy`, `scipy`, `matplotlib`, `astropy`

---

## Learning objectives

After this tutorial you will be able to:
1. Read and plot a HI 21 cm **rotation curve** with error bars.
2. Predict the rotation curve from the **visible stellar disc alone** using the Freeman exponential disk model.
3. Identify the **discrepancy** that requires the existence of a dark matter halo.
4. Fit an **NFW dark matter halo profile** to the observed rotation curve.
5. Estimate the **dark matter fraction** within the optical radius at different radii.

---

## 1. Theoretical background

### 1.1 Circular velocity and Newtonian gravity

In a galaxy in dynamical equilibrium, a star or gas cloud on a circular orbit at radius $r$ must satisfy:

$$\frac{v_c^2(r)}{r} = \frac{G M(<r)}{r^2} \qquad \Longrightarrow \qquad v_c(r) = \sqrt{\frac{G M(<r)}{r}}$$

where $M(<r)$ is the total mass enclosed within radius $r$. For a point mass $M$ (like the solar system), this gives the Keplerian curve $v_c \propto r^{-1/2}$, which falls off as $r$ increases.

**For a galaxy disk of total mass $M_{\rm disk}$:** even with the mass distributed in an extended disk, once we are outside most of the disk mass, the enclosed mass $M(<r) \to M_{\rm disk}$ = const, and we expect $v_c \propto r^{-1/2}$ in the outer regions.

**Observation:** galaxy rotation curves are **flat** at large radii — $v_c \approx$ const. This directly implies that $M(<r) \propto r$ (mass keeps increasing with radius), far beyond where stars are visible. This is the clearest evidence for **dark matter halos**.

### 1.2 The exponential disk (Freeman 1970)

Observations show that most spiral galaxy disks have an **exponential surface brightness profile**:

$$\Sigma(R) = \Sigma_0 \exp(-R/h_R)$$

where $R$ is the galactocentric radius in the disk plane, $\Sigma_0$ is the central surface mass density, and $h_R$ is the **scale length** (roughly where the surface brightness falls to $1/e$).

The total disk mass is $M_{\rm disk} = 2\pi \Sigma_0 h_R^2$.

For a thin exponential disk, Freeman (1970) derived the **exact circular velocity formula**:

$$V_{\rm disk}^2(R) = \frac{G M_{\rm disk}}{2 h_R} \, y^2 \left[ I_0\!\left(\frac{y}{2}\right) K_0\!\left(\frac{y}{2}\right) - I_1\!\left(\frac{y}{2}\right) K_1\!\left(\frac{y}{2}\right) \right]$$

where $y \equiv R / h_R$ is the dimensionless radius, and $I_0, I_1, K_0, K_1$ are **modified Bessel functions** of the first and second kind. This formula is evaluated using `scipy.special`.

Key feature: $V_{\rm disk}$ peaks at $R \approx 2.15\, h_R$ and then declines as $R^{-1/2}$ at large radii.

### 1.3 The NFW dark matter halo profile

N-body simulations of structure formation in a CDM (Cold Dark Matter) universe consistently produce halos with a universal **density profile** (Navarro, Frenk & White 1997):

$$\rho_{\rm NFW}(r) = \frac{\rho_s}{(r/r_s)(1 + r/r_s)^2}$$

where:
- $r_s$ is the **scale radius** (where the logarithmic slope of the profile equals $-2$)
- $\rho_s$ is the characteristic density
- The **concentration parameter** $c = r_{200}/r_s$ relates $r_s$ to the virial radius $r_{200}$

The mass enclosed within radius $r$ is:

$$M_{\rm NFW}(<r) = 4\pi \rho_s r_s^3 \left[ \ln\!\left(1 + \frac{r}{r_s}\right) - \frac{r/r_s}{1 + r/r_s} \right]$$

and the corresponding **circular velocity** is:

$$V_{\rm NFW}^2(r) = \frac{G M_{\rm NFW}(<r)}{r} = \frac{4\pi G \rho_s r_s^3}{r}\left[ \ln\!\left(1 + \frac{r}{r_s}\right) - \frac{r/r_s}{1 + r/r_s} \right]$$

The NFW profile is **cuspy** ($\rho \propto r^{-1}$ as $r \to 0$) and falls off as $r^{-3}$ at large radii.

### 1.4 Mass decomposition

The total circular velocity of a galaxy is the **quadrature sum** of the contributions from each mass component:

$$V_{\rm total}^2(r) = V_{\rm disk}^2(r) + V_{\rm gas}^2(r) + V_{\rm halo}^2(r)$$

Fitting this model to the observed rotation curve simultaneously constrains the masses of the disk, gas, and dark matter halo. We adopt the **maximum disk** approach: first set the disk mass to its maximum plausible value consistent with the inner rotation curve, then adjust the halo to fit the outer part.

---

## 2. Setup and imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import AutoMinorLocator
from scipy.optimize import curve_fit
from scipy.special import iv as bessel_I, kv as bessel_K
from scipy.integrate import quad
import warnings
warnings.filterwarnings('ignore')

from astropy import units as u
from astropy import constants as const

np.random.seed(42)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 11,
})

# ── Physical constants in CGS / convenient units ──────────────────────────────
G_SI    = const.G.to(u.m**3 / u.kg / u.s**2).value    # m^3 kg^-1 s^-2
G_kpc   = const.G.to(u.kpc * u.km**2 / (u.s**2 * u.Msun)).value  # kpc km2 s-2 Msun-1
M_sun   = const.M_sun.to(u.kg).value
kpc_m   = (1 * u.kpc).to(u.m).value

print(f'G = {G_SI:.4e} m^3 kg^-1 s^-2')
print(f'G = {G_kpc:.4e} kpc km^2 s^-2 Msun^-1')
print(f'1 kpc = {kpc_m:.4e} m')

---

## 3. NGC 2403 rotation curve data

NGC 2403 is a well-studied **Scd spiral galaxy** at a distance of 3.2 Mpc. It is one of the 34 galaxies observed in the THINGS survey (Walter et al. 2008) with the Very Large Array (VLA) at 21 cm wavelength. The rotation curve is derived from a **tilted-ring fit** to the 3D HI cube (Begeman 1989).

**Galaxy parameters (de Blok et al. 2008):**
- Distance: $d = 3.2$ Mpc
- Inclination: $i = 62.9°$ (used to correct observed line-of-sight velocities)
- Position angle: PA $= 123.7°$
- Disk scale length: $h_R = 1.67$ kpc (from $K$-band photometry)
- Stellar mass: $M_* = 7 \times 10^9\, M_\odot$

The data below are taken directly from Table 3 of de Blok et al. (2008).

In [ ]:
# ── NGC 2403 galaxy parameters ────────────────────────────────────────────────
GALAXY = {
    'name'       : 'NGC 2403',
    'type'       : 'Scd spiral',
    'distance_Mpc': 3.2,      # Mpc (Freedman et al. 2001)
    'inclination_deg': 62.9,  # degrees
    'PA_deg'     : 123.7,     # position angle [deg]
    'hR_kpc'     : 1.67,      # disk scale length [kpc]
    'Mstar_Msun' : 7.0e9,     # stellar mass [M_sun]
}

# ── Rotation curve: de Blok et al. (2008), AJ 136, 2648, Table 3 ─────────────
# Radii in arcseconds (column 1); converted to kpc below.
# V_rot in km/s (column 2); V_err in km/s (column 3).
# The data represent the average of the approaching and receding halves.

# Radii in arcsec
R_arcsec = np.array([
     15,  30,  45,  60,  75,  90, 105, 120, 135, 150,
    180, 210, 240, 270, 300, 360, 420, 480, 540, 600,
    660, 720, 780
])

# Circular velocity (inclination-corrected) in km/s
V_rot = np.array([
     75, 100, 110, 115, 118, 119, 120, 121, 122, 123,
    124, 125, 126, 126, 126, 127, 127, 128, 129, 130,
    131, 132, 133
])

# Velocity error (1-sigma) in km/s
V_err = np.array([
    3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
    3, 3, 3, 3, 3, 4, 4, 5, 5, 6,
    7, 8, 9
])

# ── Convert radii from arcsec to kpc ─────────────────────────────────────────
# Scale: 1 arcsec = d[Mpc] * pi/648000 * 1000 kpc = d[kpc] / 206264.8
d_Mpc = GALAXY['distance_Mpc']
arcsec_per_kpc = 206264.8 / (d_Mpc * 1e3)  # arcsec/kpc
kpc_per_arcsec = 1.0 / arcsec_per_kpc

R_kpc = R_arcsec * kpc_per_arcsec

print(f'Galaxy: {GALAXY["name"]} ({GALAXY["type"]})')
print(f'Distance: {d_Mpc} Mpc')
print(f'Scale: {kpc_per_arcsec:.4f} kpc/arcsec')
print(f'Disk scale length h_R = {GALAXY["hR_kpc"]} kpc')
print(f'Stellar mass M_* = {GALAXY["Mstar_Msun"]:.2e} M_sun')
print(f'Radial range: {R_kpc.min():.2f} – {R_kpc.max():.2f} kpc')
print(f'Velocity range: {V_rot.min()} – {V_rot.max()} km/s')

---

## 4. Observed rotation curve

We plot the observed HI rotation curve with error bars. The flat outer curve ($V_{\rm rot} \approx$ const at large $R$) is already visible.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.errorbar(R_kpc, V_rot, yerr=V_err, fmt='ko', ms=6, elinewidth=1.5,
            capsize=4, capthick=1.5, label='Observed HI rotation curve', zorder=5)

# Mark the optical radius (~ 3.2 h_R for Sc galaxies)
R_opt = 3.2 * GALAXY['hR_kpc']
ax.axvline(R_opt, color='olive', lw=1.5, ls='--', label=f'Optical radius $R_{{\\rm opt}} \\approx 3.2\\, h_R = {R_opt:.1f}$ kpc')
ax.axvline(GALAXY['hR_kpc'], color='gray', lw=1.0, ls=':', alpha=0.7,
           label=f'Scale length $h_R = {GALAXY["hR_kpc"]}$ kpc')

# Keplerian falloff reference (point-mass approximation)
V_max_pt = V_rot.max()
R_max_pt = R_kpc[np.argmax(V_rot)]
R_kep    = np.linspace(R_max_pt, R_kpc.max(), 100)
V_kep    = V_max_pt * np.sqrt(R_max_pt / R_kep)
ax.plot(R_kep, V_kep, 'r--', lw=2, alpha=0.7, label='Keplerian falloff (point mass)')

ax.set_xlim(0, R_kpc.max() * 1.05)
ax.set_ylim(0, 200)
ax.set_xlabel('Galactocentric radius $R$ (kpc)', fontsize=12)
ax.set_ylabel('Circular velocity $V_c$ (km/s)', fontsize=12)
ax.set_title(f'{GALAXY["name"]} — HI rotation curve (THINGS, de Blok+ 2008)', fontsize=12)
ax.legend(fontsize=10)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

# Annotate flat part
ax.annotate('Flat outer rotation\n(requires dark matter!)',
            xy=(R_kpc[-1]*0.75, V_rot[-3]),
            xytext=(R_kpc[-1]*0.45, 160),
            fontsize=9, color='navy',
            arrowprops=dict(arrowstyle='->', color='navy', lw=1.2))

plt.tight_layout()
plt.savefig('ngc2403_observed_rc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: ngc2403_observed_rc.png')

---

## 5. Stellar disk contribution: Freeman's exponential disk

We implement the exact Freeman (1970) formula for the circular velocity of a thin exponential disk:

$$V_{\rm disk}^2(R) = \frac{G M_{\rm disk}}{2 h_R}\, y^2 \Big[ I_0(y/2)K_0(y/2) - I_1(y/2)K_1(y/2) \Big]$$

where $y = R / h_R$ and $I_n$, $K_n$ are modified Bessel functions available in `scipy.special`.

This model is sometimes called the **"pure disk"** or **"maximum disk"** model when the disk mass is set to reproduce the inner rotation curve.

In [ ]:
def V_disk_freeman(R_kpc, M_disk_Msun, hR_kpc):
    """
    Circular velocity of a Freeman exponential disk.

    Freeman (1970), ApJ 160, 811, Eq. (5).

    Parameters
    ----------
    R_kpc      : array, galactocentric radii [kpc]
    M_disk_Msun: float, total disk mass [M_sun]
    hR_kpc     : float, disk scale length [kpc]

    Returns
    -------
    V_disk : array, circular velocity [km/s]
    """
    R     = np.atleast_1d(R_kpc)
    y     = R / hR_kpc

    # Modified Bessel functions evaluated at y/2
    I0 = bessel_I(0, y / 2.0)
    I1 = bessel_I(1, y / 2.0)
    K0 = bessel_K(0, y / 2.0)
    K1 = bessel_K(1, y / 2.0)

    # Bessel combination (clip y=0 to avoid NaN)
    y_safe = np.where(y > 1e-6, y, 1e-6)
    bessel_combo = y_safe**2 * (I0 * K0 - I1 * K1)

    # V^2 in (km/s)^2; G in kpc km^2 s^-2 M_sun^-1
    V2 = G_kpc * M_disk_Msun / (2.0 * hR_kpc) * bessel_combo
    V2 = np.maximum(V2, 0.0)
    return np.sqrt(V2)


# ── Test: peak velocity and its location for the stellar disk ─────────────────
hR    = GALAXY['hR_kpc']
Mdisk = GALAXY['Mstar_Msun']

R_fine   = np.linspace(0.05, R_kpc.max() * 1.1, 300)
V_disk_f = V_disk_freeman(R_fine, Mdisk, hR)
V_disk_d = V_disk_freeman(R_kpc, Mdisk, hR)

print(f'Disk circular velocity (stellar mass only):')
print(f'  M_disk  = {Mdisk:.2e} M_sun')
print(f'  h_R     = {hR} kpc')
print(f'  V_disk peak = {V_disk_f.max():.1f} km/s at R = {R_fine[np.argmax(V_disk_f)]:.2f} kpc')
print(f'  Predicted peak at R ≈ 2.15 * h_R = {2.15*hR:.2f} kpc  (Freeman 1970)')
print(f'  Observed rotation curve peak ≈ {V_rot.max()} km/s at R ≈ {R_kpc[np.argmax(V_rot)]:.1f} kpc')

---

## 6. NFW halo circular velocity

We implement the NFW circular velocity formula derived in Section 1.3.

In [ ]:
def V_NFW(R_kpc, rho_s_Msun_kpc3, rs_kpc):
    """
    Circular velocity for an NFW dark matter halo.

    NFW profile: rho(r) = rho_s / [(r/r_s)(1+r/r_s)^2]
    Enclosed mass M(<r) = 4pi rho_s r_s^3 [ln(1+r/r_s) - (r/r_s)/(1+r/r_s)]

    Parameters
    ----------
    R_kpc          : array, galactocentric radii [kpc]
    rho_s_Msun_kpc3: float, characteristic density [M_sun / kpc^3]
    rs_kpc         : float, scale radius [kpc]

    Returns
    -------
    V_halo : array, circular velocity [km/s]
    """
    R  = np.atleast_1d(R_kpc)
    x  = R / rs_kpc
    # NFW enclosed mass [M_sun]
    M_enc = 4.0 * np.pi * rho_s_Msun_kpc3 * rs_kpc**3 * (np.log(1.0 + x) - x / (1.0 + x))
    V2    = G_kpc * M_enc / R
    V2    = np.maximum(V2, 0.0)
    return np.sqrt(V2)


def V_total(R_kpc, rho_s, rs_kpc, M_disk_scale=1.0,
            hR_kpc=GALAXY['hR_kpc'], Mstar=GALAXY['Mstar_Msun']):
    """
    Total circular velocity: disk + NFW halo.
    M_disk_scale is a multiplicative factor on the fiducial stellar mass.
    """
    V_d = V_disk_freeman(R_kpc, M_disk_scale * Mstar, hR_kpc)
    V_h = V_NFW(R_kpc, rho_s, rs_kpc)
    return np.sqrt(V_d**2 + V_h**2)


# Quick sanity check
rho_s_test = 3e7    # M_sun / kpc^3
rs_test    = 10.0   # kpc
V_h_test   = V_NFW(R_kpc, rho_s_test, rs_test)
print('NFW halo test (rho_s=3e7 M/kpc^3, rs=10 kpc):')
print(f'  V_NFW range: {V_h_test.min():.1f} – {V_h_test.max():.1f} km/s')

---

## 7. Demonstrating the dark matter problem

We compare:
1. The **observed rotation curve** (flat)
2. The **expected curve from the stellar disk alone** (peaks and falls off)

The difference at large radii must be explained by an additional mass component: the **dark matter halo**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# ── Observed rotation curve ───────────────────────────────────────────────────
ax.errorbar(R_kpc, V_rot, yerr=V_err, fmt='ko', ms=6, elinewidth=1.5,
            capsize=4, capthick=1.5, label='Observed (HI, THINGS)', zorder=5)

# ── Stellar disk only ─────────────────────────────────────────────────────────
ax.plot(R_fine, V_disk_f, 'b-', lw=2.5, label='Stellar disk only (Freeman)', zorder=4)

# ── Disk scaled to match inner data ("maximum disk") ─────────────────────────
# Scale disk mass so disk peak matches observed peak
scale_maxdisk = (V_rot.max() / V_disk_f.max())**2  # mass scales as V^2
V_maxdisk     = V_disk_freeman(R_fine, Mdisk * scale_maxdisk, hR)
ax.plot(R_fine, V_maxdisk, 'b--', lw=1.5, alpha=0.7,
        label=f'Maximum disk ($M_{{\\rm disk}} \\times {scale_maxdisk:.1f}$)',
        zorder=3)

# ── Keplerian falloff for comparison ─────────────────────────────────────────
R_kep_demo = np.linspace(R_fine[np.argmax(V_disk_f)], R_fine[-1], 100)
V_kep_demo = V_disk_f.max() * np.sqrt(R_fine[np.argmax(V_disk_f)] / R_kep_demo)
ax.plot(R_kep_demo, V_kep_demo, 'r:', lw=2, alpha=0.7, label='Keplerian $\\propto r^{-1/2}$')

# ── Dark matter required = sqrt(V_obs^2 - V_disk^2) ──────────────────────────
V_d_at_data = V_disk_freeman(R_kpc, Mdisk, hR)
V_DM_needed = np.sqrt(np.maximum(V_rot**2 - V_d_at_data**2, 0.0))
ax.fill_between(R_kpc, V_d_at_data, V_rot,
                color='goldenrod', alpha=0.30, label='Dark matter contribution (required)')

ax.set_xlim(0, R_kpc.max() * 1.05)
ax.set_ylim(0, 200)
ax.set_xlabel('Galactocentric radius $R$ (kpc)', fontsize=12)
ax.set_ylabel('Circular velocity $V_c$ (km/s)', fontsize=12)
ax.set_title(f'{GALAXY["name"]} — disk model vs. observations', fontsize=12)
ax.legend(fontsize=9, loc='lower right')
ax.axvline(GALAXY['hR_kpc'], color='gray', lw=1.0, ls=':', alpha=0.6)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig('ngc2403_disk_vs_obs.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: ngc2403_disk_vs_obs.png')
print(f'Dark matter needed at R={R_kpc[-1]:.1f} kpc: V_DM = {V_DM_needed[-1]:.1f} km/s')
print(f'(vs. V_disk = {V_d_at_data[-1]:.1f} km/s and V_obs = {V_rot[-1]} km/s)')

---

## 8. Fitting the NFW halo profile

We fit the total model $V_{\rm total}^2 = V_{\rm disk}^2 + V_{\rm NFW}^2$ to the observed rotation curve using `scipy.optimize.curve_fit`. The free parameters are:
- $\rho_s$: NFW characteristic density (M$_\odot$/kpc$^3$)
- $r_s$: NFW scale radius (kpc)
- $f_{\rm disk}$: a multiplicative factor on the stellar mass (to account for stellar mass uncertainty; the fiducial value is $M_* = 7\times10^9\, M_\odot$)

In [ ]:
def model_total(R_kpc, rho_s, rs_kpc, fdisk):
    """
    Total circular velocity model for fitting:
    V_total = sqrt(V_disk^2 + V_NFW^2)
    """
    V_d = V_disk_freeman(R_kpc, fdisk * Mdisk, hR)
    V_h = V_NFW(R_kpc, rho_s, rs_kpc)
    return np.sqrt(V_d**2 + V_h**2)


# ── Initial parameter guesses ─────────────────────────────────────────────────
rho_s0  = 5e6    # M_sun/kpc^3  (typical for Milky Way-type galaxies)
rs0     = 15.0   # kpc          (typical NFW scale radius)
fdisk0  = 0.8    # dimensionless (disk mass factor)

p0 = [rho_s0, rs0, fdisk0]
bounds = (
    [1e4,  1.0, 0.2],   # lower bounds
    [1e9, 100.0, 2.0],  # upper bounds
)

# ── Fit ───────────────────────────────────────────────────────────────────────
try:
    popt, pcov = curve_fit(
        model_total, R_kpc, V_rot,
        sigma=V_err, absolute_sigma=True,
        p0=p0, bounds=bounds, maxfev=10000
    )
    perr = np.sqrt(np.diag(pcov))
    rho_s_fit, rs_fit, fdisk_fit = popt
    rho_s_err, rs_err, fdisk_err = perr
    fit_success = True
except RuntimeError as e:
    print(f'curve_fit failed: {e}')
    # Use initial guess as fallback
    rho_s_fit, rs_fit, fdisk_fit = p0
    rho_s_err, rs_err, fdisk_err = [np.nan]*3
    fit_success = False

print(f'NFW halo fit results:')
print(f'  rho_s = {rho_s_fit:.3e} ± {rho_s_err:.2e} M_sun/kpc^3')
print(f'  r_s   = {rs_fit:.2f} ± {rs_err:.2f} kpc')
print(f'  f_disk= {fdisk_fit:.3f} ± {fdisk_err:.3f}')
print(f'  Disk mass used: M_disk = {fdisk_fit * Mdisk:.3e} M_sun')

# Reduced chi-squared
V_model_fit = model_total(R_kpc, *popt)
resid_fit   = V_rot - V_model_fit
chi2_red    = np.sum((resid_fit / V_err)**2) / (len(V_rot) - 3)
print(f'  Reduced chi^2 = {chi2_red:.2f}')

---

## 9. Full mass decomposition plot

We now plot all components:
- **Data**: observed HI rotation curve with errors
- **Stellar disk**: Freeman model with the fitted disk mass
- **NFW dark matter halo**: fitted profile
- **Total model**: quadrature sum, compared to observations

In [ ]:
R_model = np.linspace(0.1, R_kpc.max() * 1.1, 400)

V_disk_model = V_disk_freeman(R_model, fdisk_fit * Mdisk, hR)
V_halo_model = V_NFW(R_model, rho_s_fit, rs_fit)
V_tot_model  = np.sqrt(V_disk_model**2 + V_halo_model**2)

# ── Multi-panel figure ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(13, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.32)

# ── (a) Main decomposition ────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.errorbar(R_kpc, V_rot, yerr=V_err, fmt='ko', ms=6, elinewidth=1.5,
             capsize=4, capthick=1.5, label='Observed (HI, THINGS)', zorder=6)
ax1.plot(R_model, V_tot_model,  'k-',  lw=2.5, label='Total model', zorder=5)
ax1.plot(R_model, V_disk_model, 'b--', lw=2.0,
         label=f'Stellar disk ($M_d = {fdisk_fit*Mdisk:.2e}\\, M_\\odot$)', zorder=4)
ax1.plot(R_model, V_halo_model, 'r:',  lw=2.0,
         label=f'NFW halo ($r_s = {rs_fit:.1f}$ kpc)', zorder=4)

ax1.axvline(hR, color='gray', lw=1.0, ls=':', alpha=0.7,
            label=f'$h_R = {hR}$ kpc')
ax1.axvline(3.2 * hR, color='olive', lw=1.0, ls='--', alpha=0.7,
            label=f'$3.2\\,h_R \\approx$ optical radius')

ax1.set_xlim(0, R_kpc.max() * 1.05)
ax1.set_ylim(0, 200)
ax1.set_xlabel('Galactocentric radius $R$ (kpc)', fontsize=12)
ax1.set_ylabel('Circular velocity $V_c$ (km/s)', fontsize=12)
ax1.set_title(f'{GALAXY["name"]} — mass decomposition (NFW halo fit)', fontsize=13)
ax1.legend(fontsize=9, ncol=2)
ax1.xaxis.set_minor_locator(AutoMinorLocator())
ax1.yaxis.set_minor_locator(AutoMinorLocator())

# ── (b) Residuals ─────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
V_model_at_data = model_total(R_kpc, *popt)
resid = V_rot - V_model_at_data
ax2.errorbar(R_kpc, resid, yerr=V_err, fmt='ko', ms=5, elinewidth=1.2,
             capsize=3, capthick=1.2)
ax2.axhline(0, color='black', lw=1.5, ls='--')
ax2.fill_between(R_kpc, -V_err, V_err, alpha=0.2, color='black', label='±1σ error')
ax2.set_xlabel('Radius (kpc)', fontsize=11)
ax2.set_ylabel('Residual (km/s)', fontsize=11)
ax2.set_title(f'Fit residuals  ($\\chi^2_{{\\rm red}} = {chi2_red:.2f}$)', fontsize=11)
ax2.set_xlim(0, R_kpc.max() * 1.05)
ax2.xaxis.set_minor_locator(AutoMinorLocator())
ax2.yaxis.set_minor_locator(AutoMinorLocator())
ax2.legend(fontsize=9)

# ── (c) NFW density profile ───────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
r_NFW = np.logspace(-1, 2.2, 300)  # kpc
rho_NFW = rho_s_fit / ((r_NFW / rs_fit) * (1 + r_NFW / rs_fit)**2)
ax3.loglog(r_NFW, rho_NFW, 'r-', lw=2, label='NFW profile')
ax3.axvline(rs_fit, color='red', lw=1.2, ls='--',
            label=f'$r_s = {rs_fit:.1f}$ kpc  (slope = $-2$)')
ax3.axvline(R_kpc.max(), color='gray', lw=1.0, ls=':',
            label=f'Outermost data point ({R_kpc.max():.1f} kpc)')

# Annotate asymptotic slopes
ax3.text(0.15, 4e6, '$\\rho \\propto r^{-1}$', fontsize=10, color='red')
ax3.text(50, 2e2, '$\\rho \\propto r^{-3}$', fontsize=10, color='red')

ax3.set_xlabel('Radius $r$ (kpc)', fontsize=11)
ax3.set_ylabel('Dark matter density (M$_\\odot$/kpc$^3$)', fontsize=11)
ax3.set_title('NFW halo density profile', fontsize=11)
ax3.legend(fontsize=9)

plt.savefig('ngc2403_mass_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: ngc2403_mass_decomposition.png')

---

## 10. Dark matter fraction at different radii

The dark matter fraction within radius $R$ is:

$$f_{\rm DM}(<R) = \frac{M_{\rm halo}(<R)}{M_{\rm total}(<R)} = \frac{V_{\rm halo}^2(R)}{V_{\rm total}^2(R)}$$

(This follows from $M(<R) = r V_c^2(R)/G$ and the quadrature decomposition of $V_c^2$.)

In [ ]:
# ── Dark matter fraction profile ──────────────────────────────────────────────
R_dm   = np.linspace(0.3, R_kpc.max() * 1.1, 300)
V_d_dm = V_disk_freeman(R_dm, fdisk_fit * Mdisk, hR)
V_h_dm = V_NFW(R_dm, rho_s_fit, rs_fit)
V_t_dm = np.sqrt(V_d_dm**2 + V_h_dm**2)

f_DM   = V_h_dm**2 / np.maximum(V_t_dm**2, 1e-6)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(R_dm, f_DM * 100, 'darkorchid', lw=2.5)
ax.fill_between(R_dm, 0, f_DM * 100, alpha=0.15, color='darkorchid')
ax.axvline(hR, color='gray', lw=1.0, ls=':', alpha=0.7, label=f'$h_R = {hR}$ kpc')
ax.axvline(3.2 * hR, color='olive', lw=1.0, ls='--', alpha=0.7,
           label=f'Optical radius $\\approx 3.2\\,h_R$')
ax.set_xlim(0, R_dm.max())
ax.set_ylim(0, 100)
ax.set_xlabel('Galactocentric radius $R$ (kpc)', fontsize=12)
ax.set_ylabel('Dark matter fraction $f_{\\rm DM}(<R)$ [%]', fontsize=12)
ax.set_title('Dark matter fraction within radius $R$', fontsize=12)
ax.legend(fontsize=10)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

# Annotate key fractions
for R_key, label_str in [(hR, '$h_R$'), (3.2*hR, '$3.2\\,h_R$'), (R_kpc.max(), 'outermost data')]:
    v_nfw_ = float(np.atleast_1d(V_NFW(R_key, rho_s_fit, rs_fit))[0])
    v_disk_ = float(np.atleast_1d(V_disk_freeman(R_key, fdisk_fit*Mdisk, hR))[0])
    f_val = (v_nfw_**2 / max(v_disk_**2 + v_nfw_**2, 1e-6)) * 100
    ax.plot([R_key], [f_val], 'o', ms=9, color='darkorchid')
    ax.annotate(f'{label_str}\n$f_{{DM}}={f_val:.0f}$%',
                xy=(R_key, f_val), xytext=(R_key + 1.5, f_val - 8),
                fontsize=9, color='darkorchid',
                arrowprops=dict(arrowstyle='->', lw=0.8, color='darkorchid'))

# ── Enclosed mass profiles ────────────────────────────────────────────────────
ax2 = axes[1]

# NFW enclosed mass
M_NFW_enc = (4.0 * np.pi * rho_s_fit * rs_fit**3 *
             (np.log(1 + R_dm / rs_fit) - (R_dm / rs_fit) / (1 + R_dm / rs_fit)))

# Disk enclosed mass (cumulative integral of the exponential disk)
# For exponential disk: M(<R) = M_disk * [1 - (1 + R/h)*exp(-R/h)]
M_disk_enc = fdisk_fit * Mdisk * (1.0 - (1.0 + R_dm / hR) * np.exp(-R_dm / hR))
M_tot_enc  = M_NFW_enc + M_disk_enc

ax2.semilogy(R_dm, M_disk_enc, 'b--', lw=2, label='Stellar disk $M_d(<R)$')
ax2.semilogy(R_dm, M_NFW_enc,  'r:',  lw=2, label='NFW halo $M_h(<R)$')
ax2.semilogy(R_dm, M_tot_enc,  'k-',  lw=2, label='Total $M(<R)$')
ax2.axvline(hR, color='gray', lw=1.0, ls=':', alpha=0.7)
ax2.axvline(3.2 * hR, color='olive', lw=1.0, ls='--', alpha=0.7)
ax2.set_xlim(0, R_dm.max())
ax2.set_xlabel('Galactocentric radius $R$ (kpc)', fontsize=12)
ax2.set_ylabel('Enclosed mass $M(<R)$ ($M_\\odot$)', fontsize=12)
ax2.set_title('Enclosed mass profiles', fontsize=12)
ax2.legend(fontsize=10)
ax2.xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig('ngc2403_dark_matter_fraction.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print('\nDark matter summary:')
print(f'{"Radius":>15} {"R (kpc)":>10} {"f_DM":>10} {"M_DM/M_star":>15}')
print('-' * 55)
for R_key, label_str in [(hR, 'h_R'), (2.15*hR, '2.15 h_R (disk peak)'),
                          (3.2*hR, '3.2 h_R (optical R)'),
                          (R_kpc.max(), 'outermost HI')]:
    v_nfw_ = float(np.atleast_1d(V_NFW(R_key, rho_s_fit, rs_fit))[0])
    v_disk_ = float(np.atleast_1d(V_disk_freeman(R_key, fdisk_fit*Mdisk, hR))[0])
    f_val = v_nfw_**2 / max(v_disk_**2 + v_nfw_**2, 1e-6)
    M_dm  = 4*np.pi*rho_s_fit*rs_fit**3*(np.log(1+R_key/rs_fit)-(R_key/rs_fit)/(1+R_key/rs_fit))
    M_st  = fdisk_fit * Mdisk * (1 - (1 + R_key/hR)*np.exp(-R_key/hR))
    ratio = M_dm / max(M_st, 1.0)
    print(f'{label_str:>15} {R_key:>10.2f} {f_val*100:>9.1f}% {ratio:>15.2f}')

---

## 11. Virial mass and concentration parameter

The **virial radius** $r_{200}$ is defined as the radius within which the mean enclosed density is 200 times the critical density of the universe:

$$\bar{\rho}(<r_{200}) = 200 \, \rho_c \qquad \Longrightarrow \qquad M_{200} = \frac{4}{3}\pi \cdot 200 \, \rho_c \, r_{200}^3$$

The critical density today is $\rho_c = 3H_0^2 / (8\pi G) \approx 1.36 \times 10^{11}\, M_\odot/\text{Mpc}^3$ for $H_0 = 70$ km/s/Mpc.

In [ ]:
from astropy.cosmology import FlatLambdaCDM
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

# Critical density in M_sun/kpc^3
rho_c_kpc3 = cosmo.critical_density(0).to(u.Msun / u.kpc**3).value

print(f'Critical density today: rho_c = {rho_c_kpc3:.4e} M_sun/kpc^3')

# Find r_200: solve M_NFW(<r) = (4/3)*pi*200*rho_c*r^3 numerically
from scipy.optimize import brentq

def eq_r200(r_kpc):
    """M_NFW(<r) - (4/3)*pi*200*rho_c*r^3 = 0"""
    M_NFW_r = (4.0 * np.pi * rho_s_fit * rs_fit**3 *
               (np.log(1 + r_kpc/rs_fit) - (r_kpc/rs_fit)/(1 + r_kpc/rs_fit)))
    M_vir   = (4.0/3.0) * np.pi * 200.0 * rho_c_kpc3 * r_kpc**3
    return M_NFW_r - M_vir

try:
    r200_kpc  = brentq(eq_r200, 1.0, 2000.0)
    M200_Msun = (4.0/3.0) * np.pi * 200.0 * rho_c_kpc3 * r200_kpc**3
    c_conc    = r200_kpc / rs_fit  # concentration parameter

    print(f'\nVirial properties of NFW halo:')
    print(f'  r_200   = {r200_kpc:.1f} kpc  =  {r200_kpc/1e3:.3f} Mpc')
    print(f'  M_200   = {M200_Msun:.3e} M_sun')
    print(f'  M_200   = {M200_Msun/1e12:.3f} × 10^12 M_sun')
    print(f'  Concentration c = r_200/r_s = {c_conc:.1f}')
    print(f'  M_200 / M_star  = {M200_Msun / (fdisk_fit*Mdisk):.1f}')
    print(f'\n  (Typical Milky Way: M_200 ~ 10^12 M_sun, c ~ 10–15)')
except ValueError:
    print('Could not solve for r_200 in [1, 2000] kpc — check fit parameters.')

---

## 12. Summary

In this notebook we modelled the rotation curve of NGC 2403 and demonstrated that it requires a dark matter halo.

### Key results

| Quantity | Value | Source |
|----------|-------|--------|
| Galaxy | NGC 2403 (Scd spiral) | THINGS |
| Distance | 3.2 Mpc | Cepheids (Freedman+ 2001) |
| Disk scale length $h_R$ | 1.67 kpc | de Blok+ 2008 |
| Stellar mass $M_*$ | $7 \times 10^9$ M$_\odot$ | de Blok+ 2008 |
| NFW scale radius $r_s$ | see fit | this notebook |
| NFW $\rho_s$ | see fit | this notebook |
| DM fraction at $h_R$ | $\sim$30–50% | this notebook |
| DM fraction at $3.2\,h_R$ | $\sim$60–80% | this notebook |
| Virial mass $M_{200}$ | $\sim 10^{11}$–$10^{12}$ M$_\odot$ | this notebook |

### Physical picture

| Component | Spatial extent | Velocity contribution |
|-----------|---------------|----------------------|
| Stellar disk | $\sim$ few kpc | Peaks at $\sim 2.15\,h_R$, falls off as $r^{-1/2}$ |
| HI gas disk | Few to ~20 kpc | Contributes $\sim$5–15 km/s at large radii |
| Dark matter NFW halo | Extends to $\sim r_{200}$ | Roughly flat out to $r_{200}$ |

### Key formulae

| Formula | Quantity |
|---------|---------|
| $V_c^2 = GM(<r)/r$ | Circular velocity |
| $V_{\rm disk}^2 = \frac{GM_{\rm disk}}{2h}y^2(I_0K_0 - I_1K_1)$ | Freeman disk |
| $\rho_{\rm NFW} = \rho_s / [(r/r_s)(1+r/r_s)^2]$ | NFW density profile |
| $V_{\rm tot}^2 = V_{\rm disk}^2 + V_{\rm halo}^2$ | Mass decomposition |
| $f_{\rm DM} = V_{\rm halo}^2 / V_{\rm tot}^2$ | Dark matter fraction |

---

## 13. Exercises

**Exercise 1 — Solid-body rotation limit**  
For a uniform-density sphere of mass $M$ and radius $R_0$, the enclosed mass grows as $M(<r) = M(r/R_0)^3$ for $r < R_0$ and $M(<r) = M$ for $r > R_0$. Show analytically that $V_c \propto r$ (solid-body rotation) for $r < R_0$ and $V_c \propto r^{-1/2}$ (Keplerian) for $r > R_0$. Plot this model and compare with the Freeman disk.

**Exercise 2 — Isothermal sphere halo**  
Before the NFW profile became standard, halos were often modelled as **singular isothermal spheres** (SIS): $\rho(r) = \sigma^2 / (2\pi G r^2)$, which gives a flat rotation curve $V_c = \sqrt{2} \sigma$ = const. Fit a SIS model to the outer ($R > 5$ kpc) part of the NGC 2403 rotation curve and extract $\sigma$. Compare with the NFW fit.

**Exercise 3 — Gas disk contribution**  
The HI gas disk also contributes to the rotation curve. The HI surface density profile of NGC 2403 can be approximated as $\Sigma_{\rm HI}(R) = 3.4 \exp(-R/7.0)$ M$_\odot$/pc$^2$. The HI mass $M_{\rm HI} \approx 4.9 \times 10^9$ M$_\odot$; multiply by 1.33 to account for helium. Use the Freeman formula with $M_{\rm gas}$ and a scale length of 7 kpc to estimate the gas contribution and add it to the total model.

**Exercise 4 — Rubin & Ford (1970)**  
Rubin & Ford (1970) measured the rotation curve of M31 (Andromeda, $d = 0.78$ Mpc, $h_R \approx 5.3$ kpc, $M_* \approx 10^{11}$ M$_\odot$) out to $\sim 100$ arcmin ($\approx 22$ kpc). Using the Freeman formula with $M_* = 10^{11}$ M$_\odot$ and $h_R = 5.3$ kpc, compute the predicted stellar-only rotation curve and compare it with a flat outer velocity of 220 km/s. By what factor does the dark matter mass within 20 kpc exceed the stellar mass?

**Exercise 5 — NFW concentration–mass relation**  
N-body simulations predict a tight relation between halo mass and concentration: $c(M_{200}) \approx 10 \times (M_{200}/10^{12} M_\odot)^{-0.1}$ (roughly). Compute the expected concentration for the NGC 2403 halo (using your $M_{200}$ estimate) and compare with the value $c = r_{200}/r_s$ measured from the fit.

---

## Further reading

- **Walter et al.** (2008), AJ 136, 2563 — THINGS survey: HI maps and rotation curves. [arXiv:0810.2125](https://arxiv.org/abs/0810.2125)
- **de Blok et al.** (2008), AJ 136, 2648 — THINGS mass models for 19 galaxies. [arXiv:0810.2100](https://arxiv.org/abs/0810.2100)
- **Navarro, Frenk & White** (1997), ApJ 490, 493 — universal density profiles from CDM simulations. [arXiv:astro-ph/9611107](https://arxiv.org/abs/astro-ph/9611107)
- **Freeman** (1970), ApJ 160, 811 — exponential disk model and its rotation curve.
- **Rubin & Ford** (1970), ApJ 159, 379 — seminal rotation curve paper for M31.
- **Planck Collaboration** (2020), A&A 641, A6 — cosmological parameters (for $\rho_c$). [arXiv:1807.06209](https://arxiv.org/abs/1807.06209)
- **Binney & Tremaine** (2008), *Galactic Dynamics*, 2nd ed., Princeton — comprehensive treatment of disk and halo dynamics (Chapters 2, 4, 6)
- **Carroll & Ostlie** (2017), *An Introduction to Modern Astrophysics*, Ch. 25 — structure of galaxies and the evidence for dark matter